# 365 Probabilidades · Dia #066
## Qual a probabilidade de o mesmo dado provar duas coisas opostas?

**Tipo:** Metodológico
**Data de publicação:** 2026-08-18
**Ferramenta:** Python
**Decisão analisada:** Posso confiar num número agregado? E num número separado por grupo?
**Hashtag:** #365Probabilidades #Dia066

---

### 📖 A História

Existe uma tabela que aparece em quase todo curso de estatística do mundo, e quase
sempre acompanhada da mesma história errada.

A versão que circula diz que a Universidade de Berkeley foi processada por
discriminar mulheres na pós-graduação, e que os estatísticos entraram em cena e
provaram que não havia discriminação nenhuma.

Berkeley nunca foi processada. Em entrevista, o próprio Peter Bickel contou que o
vice-decano da pós-graduação achou que a universidade **poderia** ser processada, e
por isso pediu que alguém olhasse os dados antes.

O nome também está errado, de outro jeito. Edward Simpson descreveu o fenômeno em
1951, num artigo de quatro páginas escrito quando ele ainda era aluno de pós em
Cambridge. Karl Pearson já tinha esbarrado no assunto em 1899, e Udny Yule em 1903.
O apelido "paradoxo de Simpson" só apareceu em 1972, cunhado por Colin Blyth.

O próprio Simpson achava a homenagem indevida: dizia que o crédito cabia ao Yule.

Então o caso mais famoso de leitura errada de dados vem embrulhado numa história
falsa e num nome que credita a pessoa errada. Os números, esses, existem e são
públicos.

---

### 📚 O Conceito: o que o artigo de 1951 realmente diz

O paradoxo acontece quando uma tendência aparece dentro de cada grupo e some, ou
inverte, quando os grupos são somados. As duas leituras estão aritmeticamente
corretas ao mesmo tempo.

A moral que o mundo tirou disso é "sempre desagregue". **Não é o que está escrito no
artigo original.**

Simpson usa um baralho com que um bebê brincou, deixando algumas cartas sujas. Nas
cartas sujas e nas limpas, separadamente, aparece associação entre cor e tipo de
carta. Somando as duas, a associação evapora. Ali ele diz que **a tabela somada dá a
resposta sensata**: a sujeira do bebê era ruído.

Em seguida ele repete exatamente as mesmas contagens com outros rótulos, virando
tratamento, sexo e sobrevivência. E diz o **contrário**: que o tratamento dificilmente
pode ser descartado como inútil quando beneficia homens e beneficia mulheres.

Mesmos números, resposta certa diferente, conforme o que as variáveis significam.

### O que veio depois: confundidor ou mediador

A leitura moderna vai além. Judea Pearl argumenta que o paradoxo só parece paradoxo
porque se insiste em responder com linguagem estatística uma pergunta que é causal, e
que a escolha entre somar e separar é determinada por um modelo causal explícito.

E é aí que o caso de Berkeley fica desconfortável. Se o sexo influencia a escolha do
departamento, e o departamento influencia a admissão, então **departamento é mediador,
não confundidor**. Condicionar num mediador bloqueia justamente o caminho pelo qual a
discriminação poderia estar agindo. Pearl enquadra a conclusão de Bickel como a
estimativa do efeito **direto** do sexo, obtida ao condicionar no mediador.

Aceitar essa estratificação como resposta correta é, portanto, uma decisão normativa:
é decidir que a escolha de departamento é um caminho legítimo. Os dados não dizem isso.

Este notebook calcula as duas leituras e **não elege vencedora**.

---

### 🧮 O Modelo

Contagens administrativas completas, publicadas, mais o exemplo original de 1951.
Nenhum número deste dia é estimativa.

**Fontes:**
- Bickel, P. J., Hammel, E. A. & O'Connell, J. W., 1975 · *Science* 187(4175),
  398-404 · admissões da pós-graduação de Berkeley, outono de 1973 ·
  **12.763 candidatos a 101 departamentos** · 8.442 homens com taxa de 44,2% e
  4.321 mulheres com taxa de 34,6% · seis maiores departamentos com contagens célula
  a célula: 2.691 homens com 1.198 admitidos contra 1.835 mulheres com 557 admitidas
- Simpson, E. H., 1951 · "The Interpretation of Interaction in Contingency Tables" ·
  *Journal of the Royal Statistical Society, Series B* 13(2), 238-241 · Tabelas 2, 3
  e 4, em proporções de 52
- Pearl, J. · análise causal do exemplo de Berkeley, com departamento tratado como
  mediador e a conclusão de Bickel lida como efeito direto do sexo
- Rekdal, O. B., 2014 · *Social Studies of Science* 44(4), 638-654 · lendas urbanas
  acadêmicas, categoria à qual pertence a versão do processo judicial

**Precisão obrigatória:** o viés pequeno a favor das mulheres relatado no artigo vem
do agrupamento dos 101 departamentos. Este notebook calcula sobre os **seis maiores**,
que são os que têm contagens publicadas. São bases diferentes.

**Nota metodológica sobre o fator ×0.80:** não se aplica. Registros administrativos de
admissão, população completa, sem autorrelato em lugar nenhum.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

print("Bibliotecas carregadas")

In [ ]:
# --- DADOS DA LITERATURA ---
# Bickel, Hammel & O'Connell, 1975, Science 187(4175), 398-404
# Contagens administrativas completas. Nada aqui e estimado.

# Universo completo: 101 departamentos
n_total_1973   = 12_763
n_homens_tot   = 8_442
n_mulheres_tot = 4_321
p_homens_tot   = 0.442
p_mulheres_tot = 0.346

# Seis maiores departamentos, contagens celula a celula
# [admitidos_h, rejeitados_h, admitidas_m, rejeitadas_m]
DEPTOS = {
    'A': [512, 313,  89,  19],
    'B': [353, 207,  17,   8],
    'C': [120, 205, 202, 391],
    'D': [138, 279, 131, 244],
    'E': [ 53, 138,  94, 299],
    'F': [ 22, 351,  24, 317],
}

nomes = list(DEPTOS.keys())
adm_h = np.array([DEPTOS[d][0] for d in nomes], dtype=float)
rej_h = np.array([DEPTOS[d][1] for d in nomes], dtype=float)
adm_m = np.array([DEPTOS[d][2] for d in nomes], dtype=float)
rej_m = np.array([DEPTOS[d][3] for d in nomes], dtype=float)

cand_h = adm_h + rej_h
cand_m = adm_m + rej_m
taxa_h = adm_h / cand_h
taxa_m = adm_m / cand_m

# Agregado dos seis
A_h, R_h = adm_h.sum(), rej_h.sum()
A_m, R_m = adm_m.sum(), rej_m.sum()
tot_h, tot_m = A_h + R_h, A_m + R_m

# Simpson, 1951, JRSS-B 13(2), Tabelas 2 e 4 - proporcoes de 52
# Mesmas contagens, dois conjuntos de rotulos.
# Baralho:   [figura_vermelha, comum_vermelha, figura_preta, comum_preta]
CARTAS = {'sujas': [4, 8, 3, 5], 'limpas': [2, 12, 3, 15]}
# Tratamento: [vivo_sem_trat, vivo_tratado, morto_sem_trat, morto_tratado]
CLINICO = {'homens': [4, 8, 3, 5], 'mulheres': [2, 12, 3, 15]}

aplica_fator_080 = False

print("=" * 72)
print("  DADOS - ADMISSOES NA POS-GRADUACAO DE BERKELEY, OUTONO DE 1973")
print("=" * 72)
print(f"\n  Universo completo (101 departamentos): N={n_total_1973:,}".replace(",", "."))
print(f"  -> Homens:   {n_homens_tot:,}".replace(",", ".") + f"  taxa {p_homens_tot*100:.1f}%")
print(f"  -> Mulheres: {n_mulheres_tot:,}".replace(",", ".") + f"  taxa {p_mulheres_tot*100:.1f}%")
print(f"\n  Seis maiores departamentos: N={int(tot_h+tot_m):,}".replace(",", "."))
print(f"  -> Homens:   {int(tot_h):,}".replace(",", ".") +
      f"  admitidos {int(A_h)}  taxa {A_h/tot_h*100:.1f}%")
print(f"  -> Mulheres: {int(tot_m):,}".replace(",", ".") +
      f"  admitidas {int(A_m)}  taxa {A_m/tot_m*100:.1f}%")
print(f"\n  Departamento a departamento:")
print(f"  {'':>4}  {'cand H':>7} {'taxa H':>8}  {'cand M':>7} {'taxa M':>8}   quem tem taxa maior")
for i, d in enumerate(nomes):
    maior = "MULHERES" if taxa_m[i] > taxa_h[i] else "homens"
    print(f"  {d:>4}  {int(cand_h[i]):>7} {taxa_h[i]*100:>7.1f}%  "
          f"{int(cand_m[i]):>7} {taxa_m[i]*100:>7.1f}%   {maior}")

n_favor_m = int((taxa_m > taxa_h).sum())
print(f"\n  -> Em {n_favor_m} dos {len(nomes)} departamentos a taxa feminina e MAIOR.")
print(f"  -> No agregado, a taxa masculina e "
      f"{(A_h/tot_h - A_m/tot_m)*100:.1f} pontos maior.")
print(f"\n  Fator x0.80 aplicado: {aplica_fator_080}")
print("=" * 72)

In [ ]:
# --- O MODELO ---
# Assinatura estatistica: razao de chances bruta contra razao de chances
# comum de Mantel-Haenszel, estratificada por departamento.

Z = stats.norm.ppf(0.975)


def odds_ratio_ic(a, b, c, d):
    "a=admitidos H, b=rejeitados H, c=admitidas M, d=rejeitadas M"
    orr = (a * d) / (b * c)
    ep_log = np.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
    return orr, (np.exp(np.log(orr) - Z * ep_log), np.exp(np.log(orr) + Z * ep_log)), ep_log


# 1) Bruta, no agregado dos seis departamentos
or_bruta, ic_bruta, ep_bruta = odds_ratio_ic(A_h, R_h, A_m, R_m)

# Qui-quadrado da tabela agregada 2x2
tabela_agregada = np.array([[A_h, R_h], [A_m, R_m]])
qui2, p_qui2, gl, _ = stats.chi2_contingency(tabela_agregada, correction=False)

# 2) Por departamento
ors, ics, eps = [], [], []
for i in range(len(nomes)):
    o, ic, ep = odds_ratio_ic(adm_h[i], rej_h[i], adm_m[i], rej_m[i])
    ors.append(o); ics.append(ic); eps.append(ep)
ors = np.array(ors); eps = np.array(eps)

# 3) Mantel-Haenszel: razao de chances comum, estratificada por departamento
n_i = adm_h + rej_h + adm_m + rej_m
R_i = adm_h * rej_m / n_i
S_i = rej_h * adm_m / n_i
or_mh = R_i.sum() / S_i.sum()

# Erro padrao de Robins, Breslow e Greenland
P_i = (adm_h + rej_m) / n_i
Q_i = (rej_h + adm_m) / n_i
var_log_mh = (np.sum(P_i * R_i) / (2 * R_i.sum() ** 2)
              + np.sum(P_i * S_i + Q_i * R_i) / (2 * R_i.sum() * S_i.sum())
              + np.sum(Q_i * S_i) / (2 * S_i.sum() ** 2))
ep_log_mh = np.sqrt(var_log_mh)
ic_mh = (np.exp(np.log(or_mh) - Z * ep_log_mh), np.exp(np.log(or_mh) + Z * ep_log_mh))

# 4) Teste de homogeneidade de Woolf entre os departamentos
pesos = 1 / eps ** 2
log_or_medio = np.sum(pesos * np.log(ors)) / pesos.sum()
qui2_woolf = np.sum(pesos * (np.log(ors) - log_or_medio) ** 2)
gl_woolf = len(nomes) - 1
p_woolf = stats.chi2.sf(qui2_woolf, gl_woolf)

# 5) O mecanismo: onde cada grupo se candidatou
taxa_geral_dep = (adm_h + adm_m) / n_i
ordem = np.argsort(-taxa_geral_dep)          # do mais facil para o mais dificil
faceis = ordem[:2]
dificeis = ordem[-2:]
p_h_faceis = cand_h[faceis].sum() / tot_h
p_m_faceis = cand_m[faceis].sum() / tot_m
p_h_dificeis = cand_h[dificeis].sum() / tot_h
p_m_dificeis = cand_m[dificeis].sum() / tot_m

# 6) O exemplo original de Simpson, 1951, calculado
def razao(v):
    "v = [a, b, c, d] -> (a*d)/(b*c)"
    return (v[0] * v[3]) / (v[1] * v[2])


or_sujas  = razao(CARTAS['sujas'])
or_limpas = razao(CARTAS['limpas'])
somadas   = [CARTAS['sujas'][i] + CARTAS['limpas'][i] for i in range(4)]
or_somado = razao(somadas)

# Mesmas contagens, rotulos clinicos: razao morto/vivo antes e depois do tratamento
def fator_tratamento(v):
    "v = [vivo_sem, vivo_com, morto_sem, morto_com]"
    return (v[3] / v[1]) / (v[2] / v[0])


f_homens   = fator_tratamento(CLINICO['homens'])
f_mulheres = fator_tratamento(CLINICO['mulheres'])
clin_somado = [CLINICO['homens'][i] + CLINICO['mulheres'][i] for i in range(4)]
f_somado   = fator_tratamento(clin_somado)

print("=" * 72)
print("  MODELO - A MESMA TABELA, DUAS RESPOSTAS")
print("=" * 72)
print(f"\n  1) Olhando o agregado dos seis departamentos:")
print(f"  -> Razao de chances (homens vs mulheres): {or_bruta:.2f}"
      f"  IC 95% [{ic_bruta[0]:.2f} · {ic_bruta[1]:.2f}]")
print(f"  -> Qui-quadrado = {qui2:.1f} (gl={gl}), p = {p_qui2:.2e}")
print(f"  -> Leitura: homens tem quase o dobro da chance de entrar.")
print(f"\n  2) Olhando departamento por departamento:")
for i, d in enumerate(nomes):
    direcao = "favorece mulheres" if ors[i] < 1 else "favorece homens  "
    print(f"  -> {d}: OR = {ors[i]:.2f}  IC 95% [{ics[i][0]:.2f} · {ics[i][1]:.2f}]"
          f"  {direcao}")
print(f"\n  3) Razao de chances comum de Mantel-Haenszel:")
print(f"  -> OR_MH = {or_mh:.2f}  IC 95% [{ic_mh[0]:.2f} · {ic_mh[1]:.2f}]")
cruza = "SIM" if ic_mh[0] < 1 < ic_mh[1] else "NAO"
print(f"  -> O intervalo cruza 1? {cruza}")
print(f"\n  4) Homogeneidade entre departamentos (teste de Woolf):")
print(f"  -> Qui-quadrado = {qui2_woolf:.1f} (gl={gl_woolf}), p = {p_woolf:.4f}")
print(f"\n  5) O mecanismo, em uma linha:")
print(f"  -> Nos dois departamentos mais faceis: {p_h_faceis*100:.1f}% dos homens"
      f" contra {p_m_faceis*100:.1f}% das mulheres se candidataram")
print(f"  -> Nos dois mais dificeis: {p_h_dificeis*100:.1f}% dos homens"
      f" contra {p_m_dificeis*100:.1f}% das mulheres")
print(f"\n  6) O EXEMPLO ORIGINAL DE SIMPSON, 1951 (mesmas contagens):")
print(f"  Baralho    -> sujas {or_sujas:.3f} · limpas {or_limpas:.3f}"
      f" · SOMADO {or_somado:.3f}")
print(f"  Tratamento -> homens {f_homens:.3f} · mulheres {f_mulheres:.3f}"
      f" · SOMADO {f_somado:.3f}")
print(f"  Nos dois casos a associacao existe nos estratos e some ao somar.")
print(f"  Simpson aceita o SOMADO no baralho e o SEPARADO no tratamento.")
print(f"\n  AVISO SOBRE A ESTRATIFICACAO:")
print(f"  Mantel-Haenszel supoe que departamento e CONFUNDIDOR.")
print(f"  Se departamento for MEDIADOR (sexo -> departamento -> admissao),")
print(f"  estratificar bloqueia o caminho e responde outra pergunta.")
print(f"  Os dados nao dizem qual dos dois e. A escolha e de quem analisa.")

print(f"\n  A associacao agregada nao mede o que acontece na porta.")
print(f"  Mede em qual porta cada grupo foi bater.")
print("=" * 72)

In [ ]:
# --- VISUALIZACAO ---

DOURADO = '#c8a84b'
VERMELHO = '#c0392b'
VERDE = '#2a8a82'
CINZA = '#6b6a64'

# GRAFICO 1 - O paradoxo em duas leituras
fig1, (ax1a, ax1b) = plt.subplots(1, 2, figsize=(14, 7),
                                  gridspec_kw={'width_ratios': [1, 2.4]})

ax1a.bar(['Homens', 'Mulheres'], [A_h / tot_h * 100, A_m / tot_m * 100],
         color=[VERMELHO, VERDE], alpha=0.9, width=0.55)
for x, v in zip([0, 1], [A_h / tot_h * 100, A_m / tot_m * 100]):
    ax1a.text(x, v + 1.2, f'{v:.1f}%'.replace('.', ','), ha='center',
              fontsize=19, fontweight='bold')
ax1a.set_ylim(0, 100)
ax1a.set_ylabel('Taxa de admissão (%)')
ax1a.set_title('Somando tudo', fontsize=15, pad=12)
ax1a.text(0.5, 92, f'diferença de\n{(A_h/tot_h - A_m/tot_m)*100:.1f} pontos'
          .replace('.', ','), ha='center', fontsize=12, color='#333', style='italic')

x = np.arange(len(nomes))
larg = 0.38
ax1b.bar(x - larg / 2, taxa_h * 100, larg, color=VERMELHO, alpha=0.9, label='Homens')
ax1b.bar(x + larg / 2, taxa_m * 100, larg, color=VERDE, alpha=0.9, label='Mulheres')
for i in range(len(nomes)):
    if taxa_m[i] > taxa_h[i]:
        ax1b.text(i, max(taxa_h[i], taxa_m[i]) * 100 + 3, '▲', ha='center',
                  fontsize=15, color=VERDE)
ax1b.set_xticks(x)
ax1b.set_xticklabels(nomes)
ax1b.set_xlabel('Departamento (os seis maiores)')
ax1b.set_ylim(0, 100)
ax1b.set_ylabel('Taxa de admissão (%)')
ax1b.set_title(f'Separando por departamento: em {n_favor_m} dos {len(nomes)} a taxa '
               f'feminina é maior', fontsize=15, pad=12)
ax1b.legend(frameon=False, fontsize=12)

fig1.suptitle('A mesma tabela, duas conclusões opostas\n'
              'Bickel, Hammel & O\'Connell, 1975, Science 187(4175)',
              fontsize=16, y=1.03)
plt.figtext(0.5, 0.005,
            'Fonte: Bickel et al., 1975, Science  |  #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-066-grafico-01-paradoxo.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 1 salvo")

# GRAFICO 2 - O mecanismo: onde cada grupo bateu na porta
fig2, ax2 = plt.subplots(figsize=(12, 8))

prop_mulheres = cand_m / (cand_h + cand_m) * 100
ax2.scatter(taxa_geral_dep * 100, prop_mulheres,
            s=n_i * 0.55, color=DOURADO, alpha=0.55, edgecolor='#8a6f2a', linewidth=2)
for i, d in enumerate(nomes):
    ax2.text(taxa_geral_dep[i] * 100, prop_mulheres[i], d, ha='center', va='center',
             fontsize=17, fontweight='bold', color='#3a2f10')

ax2.set_xlabel('Taxa de admissão do departamento, homens e mulheres juntos (%)')
ax2.set_ylabel('Proporção de mulheres entre os candidatos (%)')
ax2.set_title('O mecanismo: as mulheres se candidatavam mais aos departamentos\n'
              'que rejeitavam mais gente, de qualquer sexo',
              fontsize=15, pad=18)
ax2.set_xlim(0, 75)
ax2.set_ylim(0, 80)
ax2.text(0.98, 0.97,
         f'Nos dois departamentos mais fáceis:\n'
         f'{p_h_faceis*100:.0f}% dos homens contra {p_m_faceis*100:.0f}% das mulheres\n\n'
         f'Nos dois mais difíceis:\n'
         f'{p_h_dificeis*100:.0f}% dos homens contra {p_m_dificeis*100:.0f}% das mulheres',
         transform=ax2.transAxes, ha='right', va='top', fontsize=12, color='#333',
         bbox=dict(boxstyle='round,pad=0.6', facecolor='white', edgecolor=CINZA, alpha=0.9))
ax2.text(0.02, 0.04, 'O tamanho da bolha é o número de candidatos',
         transform=ax2.transAxes, fontsize=10, color=CINZA, style='italic')

plt.figtext(0.5, 0.005,
            'Fonte: Bickel et al., 1975, Science  |  #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-066-grafico-02-mecanismo.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 2 salvo")

# GRAFICO 3 - Assinatura estatistica: as razoes de chances
fig3, ax3 = plt.subplots(figsize=(12, 8))

rotulos = [f'Depto {d}' for d in nomes] + ['', 'AGREGADO (bruta)',
                                           'MANTEL-HAENSZEL\n(estratificada)']
valores = list(ors) + [np.nan, or_bruta, or_mh]
intervalos = list(ics) + [None, ic_bruta, ic_mh]
cores3 = [CINZA] * len(nomes) + [None, VERMELHO, VERDE]
y = np.arange(len(valores))[::-1]

for yi, val, ic, cor in zip(y, valores, intervalos, cores3):
    if ic is None:
        continue
    lw = 3.4 if cor != CINZA else 2.2
    ms = 230 if cor != CINZA else 110
    ax3.plot([ic[0], ic[1]], [yi, yi], color=cor, linewidth=lw, solid_capstyle='round')
    for lim in ic:
        ax3.plot([lim, lim], [yi - 0.10, yi + 0.10], color=cor, linewidth=lw)
    ax3.scatter([val], [yi], s=ms, color=cor, zorder=3)
    if cor != CINZA:
        ax3.text(val, yi + 0.28, f'{val:.2f}'.replace('.', ','), ha='center',
                 fontsize=16, fontweight='bold', color=cor)

ax3.axvline(x=1.0, color='#333', linestyle='--', linewidth=1.6)
ax3.set_yticks(y)
ax3.set_yticklabels(rotulos, fontsize=11)
ax3.set_xscale('log')
ax3.set_xlim(0.15, 12)
ax3.set_xticks([0.25, 0.5, 1, 2, 4, 8])
ax3.get_xaxis().set_major_formatter(plt.FuncFormatter(
    lambda v, _: (f'{v:.2f}' if v < 1 else f'{v:.0f}').replace('.', ',')))
ax3.get_xaxis().set_minor_formatter(plt.NullFormatter())
ax3.tick_params(axis='x', which='minor', length=0)
ax3.set_xlabel('Razão de chances de admissão, homens em relação a mulheres (escala log)')
ax3.set_title('A assinatura estatística: a associação que some ao estratificar\n'
              f'OR bruta {or_bruta:.2f} contra OR de Mantel-Haenszel {or_mh:.2f}'
              .replace('.', ','),
              fontsize=15, pad=18)
ax3.text(1.06, y.max() - 0.2, 'OR = 1\nnenhuma diferença', fontsize=10, color='#333')
ax3.text(0.36, 0.60,
         'Mantel-Haenszel supõe departamento como CONFUNDIDOR.\n'
         'Se for MEDIADOR, esta estimativa responde outra pergunta.',
         transform=ax3.transAxes, fontsize=11, color='#333', style='italic',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor=CINZA,
                   alpha=0.92))
ax3.text(0.5, -0.135,
         f'Teste de homogeneidade de Woolf: qui-quadrado = {qui2_woolf:.1f}, '
         f'gl = {gl_woolf}, p = {p_woolf:.4f} · heterogeneidade rejeitada'
         .replace('.', ','),
         transform=ax3.transAxes, ha='center', fontsize=12, color=DOURADO,
         fontweight='bold')

plt.figtext(0.5, 0.005,
            'Fonte: Bickel et al., 1975, Science  |  #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-066-grafico-03-assinatura.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 3 salvo")

### 💡 O Insight

Some tudo: **44,5% dos homens entraram, contra 30,4% das mulheres.** Catorze pontos,
qui-quadrado de 92,2, p astronômico. Qualquer manchete escreveria "universidade
rejeita mulheres".

Separe por departamento: **em quatro dos seis, a taxa feminina é maior.**

As duas frases estão certas. Sobre a mesma tabela. Ao mesmo tempo.

O mecanismo é quase decepcionante de tão simples. Os departamentos não admitiam na
mesma proporção, e homens e mulheres não batiam nas mesmas portas. Somando tudo, o
número agregado deixa de medir o que acontecia na porta e passa a medir **em qual
porta cada grupo foi bater**.

A moral que o mundo tirou desse caso é "sempre desagregue". O artigo de 1951 diz
outra coisa, e o próprio Simpson dá respostas opostas para as mesmas contagens
conforme o que elas representam. No baralho, somar é o certo. No tratamento, separar
é o certo. A aritmética é idêntica nos dois.

E existe uma armadilha ainda maior, que quase nunca aparece quando esse caso é
contado.

**Separar por departamento também é uma decisão, e ela pode apagar exatamente o que
você queria medir.** Se o sexo influencia em qual departamento a pessoa se candidata,
e o departamento influencia a admissão, então o departamento está no meio do caminho
causal. Controlar por ele bloqueia a rota pela qual a discriminação poderia estar
agindo. A razão de Mantel-Haenszel calculada aqui só é a resposta certa **se** o
departamento for um confundidor. Se for um caminho, ela responde outra pergunta.

Nada nos dados diz qual dos dois é. Essa parte é escolha de quem analisa, e ela
precisa ser dita em voz alta.

Então este dia não conclui que não havia discriminação em Berkeley. Conclui que ela
não estava onde a manchete apontava, e que a pergunta anda para trás: por que as
mulheres se candidatavam justamente aos departamentos mais concorridos? Os próprios
autores atribuíram a disparidade a vieses sociais anteriores à candidatura, e notaram
que os departamentos mais fáceis eram os que exigiam mais matemática na formação
prévia.

Uma última coisa, e é a mais importante para quem quiser usar isso na vida prática.

O paradoxo funciona nos dois sentidos. Serve para desmontar uma denúncia legítima e
serve para esconder um efeito real, dependendo de onde a pessoa resolve cortar. Quem
só pergunta "quem foi somado com quem" diante dos números que contrariam a própria
opinião não está sendo rigoroso. Está sendo seletivo.

O dado não vem com a instrução de como lê-lo. Essa parte é sua, e ela é assumida, não
descoberta.

*Que média você aceitou este ano sem perguntar quem estava sendo somado?*

---

### ⚠️ Limitações do Modelo

- **Departamento pode ser mediador, não confundidor.** Se o sexo influencia a escolha
  do departamento, condicionar nele estima o efeito direto e apaga o efeito que passa
  pela escolha. A razão de Mantel-Haenszel deste notebook supõe confundimento. Essa
  suposição é normativa e não é testável com estes dados.
- Mesmo a leitura causal não fecha a questão: ela pressupõe ausência de confundidores
  latentes, e já se levantou a hipótese de variáveis como local de residência afetando
  tanto a escolha do departamento quanto o resultado.
- **O teste de homogeneidade de Woolf rejeita a igualdade entre departamentos**
  (p = 0,0031), puxado pelo departamento A. Resumir seis efeitos diferentes numa razão
  comum é, em alguma medida, o mesmo procedimento que este dia critica. O número está
  aqui com a ressalva colada nele.
- **Cinco dos seis intervalos por departamento cruzam 1.** O "em quatro dos seis a
  taxa feminina é maior" é contagem de sinal, não achado estatisticamente significativo
  departamento a departamento.
- Os cálculos usam os seis maiores departamentos, que são os que têm contagens
  publicadas. O artigo analisou 101, e o viés pequeno a favor das mulheres que ele
  relata vem daquele agrupamento maior.
- Razão de chances não é razão de taxas. Com taxas altas, como no departamento A, a
  razão de chances exagera a diferença em relação à diferença de proporções.
- O teste de Woolf pressupõe células suficientemente grandes. O departamento B tem
  apenas 25 candidatas, e essa estimativa é instável.
- **Isto é análise de associação em registros administrativos, não estudo causal de
  discriminação.** Nenhum modelo aqui identifica intenção, nem observa o que ocorreu
  antes da candidatura.
- O exemplo de 1951 é artificial e construído pelo autor para ilustrar o argumento.
  Não é evidência empírica de coisa nenhuma, e está aqui como objeto histórico.
- A história do processo judicial contra Berkeley é lenda. Berkeley temeu ser
  processada e por isso encomendou a análise.
- Fator ×0.80 não aplicado: registros administrativos completos, sem autorrelato.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
